### 종합 실습 과제: 최적의 하이퍼파라미터 조합을 찾아라!

이제 이론과 기법들을 총동원하여 모델의 성능을 극한까지 끌어올릴 시간입니다.

`과제 목표`: Day 2-Part 1에서 만든 `AdvancedClassifier` 모델의 테스트 정확도를 하이퍼파라미터 튜닝을 통해 `최대한 높여보세요.`

`요구사항:`

1.  아래에 정의된 하이퍼파라미터 탐색 공간(`param_grid`) 내에서 `그리드 탐색(Grid Search)`을 수행하는 코드를 완성하세요.
2.  각 조합에 대해 모델을 훈련하고 `테스트 데이터셋에 대한 정확도`를 측정 및 기록하세요.
3.  모든 조합의 테스트가 끝나면, 가장 높은 정확도를 보인 `최고의 하이퍼파라미터 조합`과 그때의 `테스트 정확도`를 출력하세요.
4.  (도전 과제) 결과를 보기 쉽게 DataFrame으로 정리하고, 정확도를 기준으로 내림차순 정렬하여 상위 5개 조합을 출력해보세요.

아래의 Starter Code를 바탕으로 과제를 완성해 보세요!

In [1]:
# [기본 라이브러리 및 데이터 준비 코드]
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import itertools

# 0. 데이터 준비
X, y = load_breast_cancer(return_X_y=True)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

class BreastCancerDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = BreastCancerDataset(X_train, y_train)
val_dataset = BreastCancerDataset(X_val, y_val)
test_dataset = BreastCancerDataset(X_test, y_test)

input_features = X_train.shape[1]
num_classes = 2

In [2]:
# [과제용 Starter Code]

# 1. 하이퍼파라미터 탐색 공간 정의
param_grid = {
    'lr': [0.01, 0.001, 0.0001],
    'batch_size': [16, 32, 64],
    'optimizer': ['Adam', 'SGD'],
    'dropout_p': [0.3, 0.5]
}

# 2. 모델 정의 (AdvancedClassifier 재사용)
class AdvancedClassifier(nn.Module):
    def __init__(self, num_features, num_classes, dropout_p=0.4):
        super(AdvancedClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(32, num_classes)
        )
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        return self.net(x)

In [3]:
results = []
best_accuracy = 0.0
best_params = {}

In [ ]:
# 3. 그리드 탐색 루프 구현
# HINT: itertools.product를 사용하면 모든 조합을 쉽게 생성할 수 있습니다.
grid = list(itertools.product(*param_grid.values()))
print(f"총 {len(grid)}개의 하이퍼파라미터 조합을 테스트합니다.")

for i, params in enumerate(grid):
    # 딕셔너리 형태로 파라미터 재구성
    p = dict(zip(param_grid.keys(), params))
    print(f"\n--- {i+1}/{len(grid)} 번째 조합 테스트: {p} ---")
    
    # ====================== 과제 영역 시작 ======================
    # TODO 1: 하이퍼파라미터에 따라 DataLoader와 모델, 옵티마이저를 설정하세요.
    batch_size = p['batch_size']
    lr = p['lr']
    dropout_p = p['dropout_p']
    optimizer_name = p['optimizer']

    # DataLoader 생성
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 모델 생성
    model = AdvancedClassifier(num_features=X_train.shape[1], num_classes=len(np.unique(y_train)), dropout_p=dropout_p)
    model = model.to(device)

    # 옵티마이저 설정
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    
    # TODO 2: 모델 학습 루프를 작성하세요. (약 50 에포크)
    # 실습 노트북을 참고하여 작성
    num_epochs = 50
    for epoch in range(num_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()

    
    # TODO 3: 학습이 끝난 모델로 테스트 데이터셋의 정확도를 계산하세요.
    # 실습 노트북을 참고하여 작성
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb

    # TODO 4: 결과를 기록하고, 최고 성능 모델인지 확인 및 업데이트하세요.
    # 실습 노트북을 참고하여 작성


    # TODO 3: 학습이 끝난 모델로 테스트 데이터셋의 정확도를 계산하세요.
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            outputs = model(xb)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == yb).sum().item()
            total += yb.size(0)
    accuracy = 100.0 * correct / total

    # TODO 4: 결과를 기록하고, 최고 성능 모델인지 확인 및 업데이트하세요.
    result = {
        'batch_size': batch_size,
        'lr': lr,
        'dropout_p': dropout_p,
        'optimizer': optimizer_name,
        'accuracy': accuracy
    }
    results.append(result)
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_params = p.copy()
        

# 4. 최종 결과 출력
print("================ 최종 결과 ================")
print(f"최고 정확도: {best_accuracy:.2f}%")
print(f"최적 하이퍼파라미터: {best_params}")

# 5. (도전 과제) 결과를 DataFrame으로 출력
results_df = pd.DataFrame(results)
print("--- 상위 5개 결과 ---")
print(results_df.sort_values(by='accuracy', ascending=False).head(5))

## Optuna를 사용한 베이지안 최적화 구현

이번 과제에서는 Optuna를 사용하여 하이퍼파라미터 최적화를 구현해보겠습니다. 

Optuna는 베이지안 최적화를 통해 효율적으로 최적의 하이퍼파라미터를 찾을 수 있게 해줍니다.

### 구현할 내용:
1. Optuna objective 함수 정의
2. 베이지안 최적화 실행
3. 최적화 과정 시각화
4. 최적 하이퍼파라미터로 최종 모델 학습

이 방법은 그리드 서치보다 훨씬 효율적이며, 특히 하이퍼파라미터 공간이 클 때 효과적입니다.


In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ====================== 과제 영역 시작 ======================
# TODO 1: Optuna objective 함수를 완성하세요.
# trial.suggest_* 메서드를 사용하여 하이퍼파라미터를 정의하고,
# 모델을 생성하여 학습한 후 검증 정확도를 반환하세요.
def objective(trial):
    # 하이퍼파라미터 정의
    # TODO: trial.suggest_* 메서드를 사용하여 하이퍼파라미터를 정의하세요
    # - hidden_size: 32~256 범위의 정수
    # - num_layers: 1~4 범위의 정수  
    # - learning_rate: 1e-4~1e-1 범위의 실수 (로그 스케일)
    # - batch_size: [16, 32, 64, 128] 중 선택
    # - dropout_rate: 0.1~0.5 범위의 실수
    
    
    # TODO: 위에서 정의한 하이퍼파라미터를 사용하여 모델을 생성하세요
    # AdvancedClassifier 클래스를 사용하세요
    
    
    # TODO: 데이터로더를 생성하세요
    # TensorDataset과 DataLoader를 사용하세요
    
    
    # TODO: 옵티마이저와 손실 함수를 정의하세요
    # Adam 옵티마이저와 CrossEntropyLoss를 사용하세요
    
    
    # TODO: 모델 학습 루프를 작성하세요 (20 에포크)
    # 실습 노트북을 참고하여 작성하세요
    
    
    # TODO: 검증 성능을 평가하세요
    # 테스트 데이터셋으로 정확도를 계산하고 반환하세요
    
    
    return accuracy

# TODO 2: Optuna study를 생성하고 최적화를 실행하세요
# direction='maximize'로 설정하고 30번의 trial을 실행하세요


# TODO 3: 최적 하이퍼파라미터와 최고 정확도를 출력하세요
# study.best_params와 study.best_value를 사용하세요


# ====================== 과제 영역 끝 ======================

In [ ]:

# ====================== 과제 영역 시작 ======================
# TODO 4: 최적화 과정을 시각화하세요
# Optuna study의 trials를 사용하여 다음을 구현하세요:
# 1. 정확도 변화 그래프 (파란색 선 + 마커)
# 2. 최고 정확도 추적 그래프 (빨간색 점선)
# 3. 하이퍼파라미터 변화 그래프 (다양한 색상)

# TODO: trials에서 정확도 값들을 추출하세요
# values = [trial.value for trial in trials]


# TODO: 각 trial의 하이퍼파라미터 값들을 수집하세요
# params_history = []
# for trial in trials:
#     params_history.append(list(trial.params.values()))


# TODO: plotly의 make_subplots를 사용하여 2x1 서브플롯을 생성하세요
# subplot_titles는 ('최적화 과정', '하이퍼파라미터 변화')로 설정하세요


# TODO: 첫 번째 서브플롯에 정확도 변화 그래프를 추가하세요
# go.Scatter를 사용하여 파란색 선 + 마커로 표시하세요


# TODO: 최고 정확도 추적 그래프를 추가하세요
# np.maximum.accumulate를 사용하여 누적 최대값을 계산하고
# 빨간색 점선으로 표시하세요


# TODO: 두 번째 서브플롯에 하이퍼파라미터 변화 그래프를 추가하세요
# 각 하이퍼파라미터마다 다른 색상을 사용하여 표시하세요


# TODO: 그래프 레이아웃을 설정하세요
# title, height, showlegend 등을 설정하세요


# TODO: x축과 y축 제목을 설정하세요


# TODO: 그래프를 표시하세요


# ====================== 과제 영역 끝 ======================

In [ ]:
# ====================== 과제 영역 시작 ======================
# TODO 5: 최적화 결과 분석 및 최종 모델 학습
# 다음 단계들을 구현하세요:
# 1. 하이퍼파라미터 중요도 분석
# 2. 최적화 과정 요약 통계
# 3. 최적 하이퍼파라미터로 최종 모델 학습
# 4. 최종 성능 평가

# TODO: Optuna의 importance 모듈을 사용하여 하이퍼파라미터 중요도를 계산하세요
# importance = optuna.importance.get_param_importances(study)


# TODO: 중요도를 출력하세요
# for param, imp in importance.items():
#     print(f"{param}: {imp:.4f}")


# TODO: 최적화 과정 요약 통계를 출력하세요
# 총 시도 횟수, 최적화 시간, 평균 시도 시간을 계산하여 출력하세요


# TODO: 최적 하이퍼파라미터로 AdvancedClassifier 모델을 생성하세요
# study.best_params를 사용하여 모델을 초기화하세요


# TODO: 최적화된 하이퍼파라미터로 옵티마이저와 데이터 로더를 설정하세요
# Adam 옵티마이저와 DataLoader를 생성하세요


# TODO: 최종 모델 학습을 구현하세요
# 100 에포크 동안 모델을 학습시키세요
# 각 에포크마다 배치 단위로 학습을 수행하세요


# TODO: 최종 성능 평가를 구현하세요
# 테스트 데이터셋에 대해 모델의 정확도를 계산하세요
# torch.no_grad()를 사용하여 메모리 효율성을 고려하세요


# TODO: 최종 정확도를 출력하세요
# final_accuracy = 100 * correct / total


# ====================== 과제 영역 끝 ======================
